# Preparing the data for training

First we indentify our needs :
- multilangual descriptions ( FR / EN )
- relatively small training dataset 1100 samples ( split into training, validation, test )
- Tech names

## 1. Tokenizer

We have two different approches:
- Using two different tokenizers, one specifically for english and the other for french but we will need to precise the language before passing it to the model ( for the tokenizer choice )
- Using a cross-lingual pretrained model

In [3]:
from transformers import AutoTokenizer, AutoModel
import json

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

DATASET_PATH = "../data/cleaned/dataset_corrected.jsonl"


c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
with open(DATASET_PATH,"r",encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

In [8]:
text = data[0].get("text")
text

"Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines."

In [10]:
tokens = tokenizer.tokenize(text)
print(tokens)

['▁Design', '▁and', '▁Dev', '▁Col', 'lab', '▁Certifica', 'te', '.', '▁Award', 'ed', '▁the', "▁'", 'Design', '▁and', '▁Dev', '▁Col', 'lab', "'", '▁certificate', '▁for', '▁participat', 'ing', '▁in', '▁a', '▁collabora', 'tive', '▁hack', 'a', 'thon', '▁focused', '▁on', '▁', 'brid', 'ging', '▁UI', '/', 'UX', '▁design', '▁and', '▁front', 'end', '▁development', '.', '▁Con', 'tribute', 'd', '▁as', '▁a', '▁Front', 'end', '▁Developer', ',', '▁implement', 'ing', '▁responsive', '▁interface', 's', '▁using', '▁Re', 'act', '▁and', '▁modern', '▁web', '▁technologies', '▁while', '▁collabora', 'ting', '▁close', 'ly', '▁with', '▁designer', 's', '▁to', '▁deliver', '▁a', '▁functional', '▁and', '▁user', '-', 'center', 'ed', '▁solution', '▁under', '▁tight', '▁deadline', 's', '.']


As you can tell, XLM-R uses a subword units tokenization style. It breaks each word into smaller units for better understanding of complex names and cross-lingual texts. 

This is very usefull in our usecase speciffically for Tech names like PostgreSQL, TensorFlow ...

__ Means the start of a new word 

In [11]:
encoded = tokenizer(text)
print(encoded)

{'input_ids': [0, 11724, 136, 40317, 11554, 6114, 83991, 67, 5, 60992, 297, 70, 242, 128640, 136, 40317, 11554, 6114, 25, 163769, 100, 42938, 214, 23, 10, 57119, 4935, 85526, 11, 50828, 162393, 98, 6, 100269, 9966, 111481, 64, 71609, 4331, 136, 12912, 3611, 34754, 5, 1657, 191145, 71, 237, 10, 43643, 3611, 152774, 4, 29479, 214, 226874, 101758, 7, 17368, 853, 47013, 136, 5744, 1467, 118299, 12960, 57119, 1916, 20903, 538, 678, 51517, 7, 47, 75060, 10, 123309, 136, 38937, 9, 30090, 297, 29806, 1379, 107137, 157206, 7, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


This is the conversion of our human text into tensors the model understands, with the numeric vocabulary IDs correcponding to tokens, and the attention_mask which tells the model which tokes are real,

for now we only have 1s across the attention_mask

but later which batching we will have padding token which will be refered to with 0.

In [12]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

['<s>', '▁Design', '▁and', '▁Dev', '▁Col', 'lab', '▁Certifica', 'te', '.', '▁Award', 'ed', '▁the', "▁'", 'Design', '▁and', '▁Dev', '▁Col', 'lab', "'", '▁certificate', '▁for', '▁participat', 'ing', '▁in', '▁a', '▁collabora', 'tive', '▁hack', 'a', 'thon', '▁focused', '▁on', '▁', 'brid', 'ging', '▁UI', '/', 'UX', '▁design', '▁and', '▁front', 'end', '▁development', '.', '▁Con', 'tribute', 'd', '▁as', '▁a', '▁Front', 'end', '▁Developer', ',', '▁implement', 'ing', '▁responsive', '▁interface', 's', '▁using', '▁Re', 'act', '▁and', '▁modern', '▁web', '▁technologies', '▁while', '▁collabora', 'ting', '▁close', 'ly', '▁with', '▁designer', 's', '▁to', '▁deliver', '▁a', '▁functional', '▁and', '▁user', '-', 'center', 'ed', '▁solution', '▁under', '▁tight', '▁deadline', 's', '.', '</s>']


`<s>` : Start of sentence token.
Used for sentence-level understanding.
`</s>` : End of sentence token.

In [13]:
samples = [
    "TensorFlow",
    "PyTorch",
    "React.js",
    "Node.js",
    "PostgreSQL",
    "FastAPI",
    "AWS Lambda",
    "Docker Compose",
    "Kubernetes"
]

for s in samples:
    print("\n", s)
    print(tokenizer.tokenize(s))


 TensorFlow
['▁Ten', 'sor', 'F', 'low']

 PyTorch
['▁Py', 'Tor', 'ch']

 React.js
['▁Re', 'act', '.', 'js']

 Node.js
['▁No', 'de', '.', 'js']

 PostgreSQL
['▁Post', 'gre', 'SQL']

 FastAPI
['▁Fast', 'API']

 AWS Lambda
['▁A', 'WS', '▁Lamb', 'da']

 Docker Compose
['▁Do', 'cker', '▁Comp', 'ose']

 Kubernetes
['▁Kub', 'erne', 'tes']


This is a full showcase of XLM-R tokenisation power, for example PostgreSQL was split into Post gre SQL which makes the model understand SQL as a seperated information, same goes for No de . js ( for js )

In [14]:
text_fr = "Développement d'une application avec React et Python"

tokens_fr = tokenizer.tokenize(text_fr)
print(tokens_fr)

['▁D', 'ével', 'opp', 'ement', '▁d', "'", 'une', '▁application', '▁avec', '▁Re', 'act', '▁et', '▁Python']


We can see that french language is perfectlly understood by the model

In [15]:
text = "React with Node.js"

enc = tokenizer(
    text,
    return_offsets_mapping=True
)

for tok_id, offset in zip(enc["input_ids"], enc["offset_mapping"]):
    token = tokenizer.convert_ids_to_tokens(tok_id)
    print(token, offset)

<s> (0, 0)
▁Re (0, 2)
act (2, 5)
▁with (6, 10)
▁No (11, 13)
de (13, 15)
. (15, 16)
js (16, 18)
</s> (0, 0)


Perfect for annotation purposes.